# 📰 Rapport d'Analyse — News Scraper

Ce notebook charge les données produites par `analyse.py` et génère des visualisations interactives pour interpréter les tendances des actualités.

---

**Structure du notebook :**
1. Chargement des données JSON
2. Aperçu des articles scrappés
3. Visualisation des mots les plus fréquents
4. Nuage de mots interactif
5. Interprétation et conclusions


## 0. Imports et configuration

On commence par importer toutes les bibliothèques nécessaires.

In [ ]:
# ── Bibliothèques standard ──────────────────────────────
import json
import os
from pathlib import Path
from collections import Counter
from datetime import datetime

# ── Bibliothèques de données et visualisation ───────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# WordCloud (optionnel)
try:
    from wordcloud import WordCloud
    WC_OK = True
except ImportError:
    WC_OK = False
    print("[INFO] wordcloud non installé. Section nuage de mots ignorée.")

# ── Configuration chemins ───────────────────────────────
# On remonte à la racine du projet depuis /notebooks ou la racine
BASE_DIR   = Path(".").resolve()
DATA_FILE  = BASE_DIR / "data" / "news_data.json"
OUTPUT_DIR = BASE_DIR / "output"

# Style des graphiques
plt.style.use("dark_background")
plt.rcParams["font.family"] = "monospace"

print(f"Répertoire de travail : {BASE_DIR}")
print(f"Fichier de données    : {DATA_FILE}")
print(f"Existe ?              : {DATA_FILE.exists()}")

## 1. Chargement des données

Le fichier `data/news_data.json` a été généré par `analyse.py`.  
On le charge avec la bibliothèque `json` et on le convertit en DataFrame pandas.

In [ ]:
# ── Chargement du JSON ──────────────────────────────────
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {DATA_FILE}\n"
        "Lancez d'abord : python scripts/analyse.py"
    )

with open(DATA_FILE, encoding="utf-8") as f:
    raw = json.load(f)

# Extraction des parties utiles
articles   = raw["articles"]
top_words  = raw["top_words"]       # dict {mot: fréquence}
generated  = raw["generated_at"]
n_articles = raw["total_articles"]

print(f"Données chargées avec succès !")
print(f"  Généré le   : {generated[:16].replace('T', ' à ')}")
print(f"  Articles    : {n_articles}")
print(f"  Mots uniques (top 50 stockés) : {len(top_words)}")

## 2. Aperçu des articles scrappés

On affiche les premiers articles sous forme de tableau pour vérifier la qualité du scraping.

In [ ]:
# ── Création du DataFrame ───────────────────────────────
df = pd.DataFrame(articles)

# Aperçu des colonnes disponibles
print("Colonnes disponibles :", list(df.columns))
print(f"Nombre de lignes     : {len(df)}")
print()

# Affichage des 5 premiers articles
df[["title", "summary", "source"]].head(5).style \
    .set_caption("Extrait des articles scrappés") \
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

In [ ]:
# ── Statistiques de base ────────────────────────────────
df["title_length"]   = df["title"].str.len()
df["summary_length"] = df["summary"].str.len()

print("Statistiques sur la longueur des textes :")
df[["title_length", "summary_length"]].describe().round(1)

## 3. Top 10 des mots les plus fréquents

### 3.1 Tableau

In [ ]:
# ── Préparation du DataFrame des mots ──────────────────
df_words = pd.DataFrame(
    list(top_words.items()),
    columns=["Mot", "Fréquence"]
).sort_values("Fréquence", ascending=False).reset_index(drop=True)

df_words.index += 1   # Rang commence à 1
df_words.index.name = "Rang"

top10_df = df_words.head(10)

# Affichage stylisé
top10_df.style \
    .background_gradient(subset=["Fréquence"], cmap="Blues") \
    .set_caption("Top 10 des mots les plus fréquents")

### 3.2 Histogramme (barres horizontales)

In [ ]:
top10 = df_words.head(10)
words  = top10["Mot"].tolist()
counts = top10["Fréquence"].tolist()

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor("#0f0f1a")
ax.set_facecolor("#0f0f1a")

colors = plt.cm.cool([i / 10 for i in range(10)])
bars = ax.barh(words[::-1], counts[::-1], color=colors[::-1],
               edgecolor="#ffffff22", linewidth=0.5)

for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            str(count), va="center", ha="left",
            color="white", fontsize=9, fontweight="bold")

ax.set_title("Top 10 mots les plus fréquents",
             color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Fréquence", color="#aaaacc")
ax.tick_params(colors="white", labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_color("#333355")
ax.spines["left"].set_color("#333355")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "histogram_notebook.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Graphique sauvegardé.")

### 3.3 Diagramme en secteurs (pie chart) — top 6 mots

In [ ]:
top6 = df_words.head(6)

fig, ax = plt.subplots(figsize=(7, 7))
fig.patch.set_facecolor("#0f0f1a")
ax.set_facecolor("#0f0f1a")

colors_pie = plt.cm.cool([i / 6 for i in range(6)])

wedges, texts, autotexts = ax.pie(
    top6["Fréquence"],
    labels=top6["Mot"],
    autopct="%1.1f%%",
    colors=colors_pie,
    startangle=140,
    wedgeprops={"edgecolor": "#0f0f1a", "linewidth": 2},
)

for text in texts:
    text.set_color("white")
    text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(9)

ax.set_title("Répartition des 6 mots dominants",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Nuage de mots

> **Qu'est-ce qu'un nuage de mots ?**  
> Un nuage de mots est une représentation visuelle où la taille de chaque mot est proportionnelle à sa fréquence. Plus un mot est grand, plus il apparaît souvent dans les articles analysés.

In [ ]:
if WC_OK:
    # On recrée le texte à partir des fréquences stockées
    text = " ".join(
        word for word, freq in top_words.items() for _ in range(freq)
    )

    wc = WordCloud(
        width=1200, height=500,
        background_color="#0f0f1a",
        colormap="cool",
        max_words=60,
        prefer_horizontal=0.8,
        collocations=False,
    ).generate(text)

    fig, ax = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("#0f0f1a")
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title("Nuage de mots des actualités",
                 color="white", fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "wordcloud_notebook.png", dpi=150,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print("Nuage de mots sauvegardé.")
else:
    print("wordcloud non disponible. Installez-le avec : pip install wordcloud")

## 5. Interprétation des résultats

Cette section analyse automatiquement les mots dominants pour en déduire les grands thèmes des actualités du jour.

In [ ]:
# ── Détection des thèmes ────────────────────────────────
THEMES = {
    "🏛️ Politique":     {"government","president","minister","party","election","vote","senate"},
    "💰 Économie":      {"economy","market","price","inflation","trade","growth","bank","stock"},
    "⚔️ Conflit":       {"war","attack","military","troops","conflict","ukraine","russia","killed"},
    "🏥 Santé":         {"health","hospital","disease","vaccine","covid","cancer","treatment"},
    "🤖 Technologie":   {"ai","technology","data","digital","cyber","software","tech","model"},
    "🌍 Environnement": {"climate","weather","fire","flood","carbon","energy","environment"},
    "⚽ Sport":         {"game","team","player","win","match","league","championship","olympic"},
}

top_words_set = set(df_words.head(20)["Mot"].tolist())

print("=" * 55)
print("  INTERPRÉTATION DES ACTUALITÉS")
print("=" * 55)
print()
print(f"  Basé sur le Top {min(20, len(top_words_set))} mots les plus fréquents.")
print()

found_any = False
for theme, keywords in THEMES.items():
    matches = keywords.intersection(top_words_set)
    if matches:
        found_any = True
        print(f"  {theme}")
        print(f"    Mots-clés détectés : {', '.join(sorted(matches))}")
        print()

if not found_any:
    print("  Aucun thème dominant clairement identifié.")
    print("  Les actualités semblent très variées ce jour-ci.")

print("=" * 55)
print()

# Source
if articles:
    print(f"  Source analysée : {articles[0].get('source', 'N/A')}")
    print(f"  Analyse du      : {generated[:10]}")

## 6. Évolution de la fréquence (courbe cumulée)

Ce graphique montre comment les fréquences décroissent rapidement après les premiers mots — c'est la **loi de Zipf** : en langage naturel, quelques mots sont très fréquents, et la grande majorité ne l'est presque pas.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor("#0f0f1a")
ax.set_facecolor("#0f0f1a")

freqs = df_words["Fréquence"].values[:30]
ranks = range(1, len(freqs) + 1)

ax.plot(ranks, freqs, color="#00e5ff", linewidth=2, marker="o",
        markersize=5, markerfacecolor="#aa00ff", markeredgewidth=0)
ax.fill_between(ranks, freqs, alpha=0.15, color="#00e5ff")

ax.set_title("Distribution des fréquences (loi de Zipf)",
             color="white", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Rang du mot", color="#aaaacc")
ax.set_ylabel("Fréquence", color="#aaaacc")
ax.tick_params(colors="white")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_color("#333355")
ax.spines["left"].set_color("#333355")

# Annoter le 1er mot
ax.annotate(f'« {df_words.iloc[0]["Mot"]} »',
            xy=(1, freqs[0]),
            xytext=(3, freqs[0] * 0.85),
            color="#ffeb3b", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#ffeb3b", lw=1.5))

plt.tight_layout()
plt.show()

---

## ✅ Conclusion

Ce notebook a permis de :

| Étape | Résultat |
|-------|----------|
| Chargement des données | ✅ JSON chargé et validé |
| Aperçu des articles | ✅ Tableau pandas affiché |
| Histogramme | ✅ Top 10 visualisé |
| Pie chart | ✅ Répartition des 6 premiers |
| Nuage de mots | ✅ (si wordcloud installé) |
| Interprétation | ✅ Thèmes détectés automatiquement |
| Loi de Zipf | ✅ Distribution illustrée |

---

*Généré par le projet News Analyser — Python + BeautifulSoup + pandas + matplotlib*